# Hybrid OCR + Geometry + VLM  →  Financial Table HTML

Direct `image → HTML` with one VLM prompt keeps failing on complex financial tables:
one early mistake (a shifted column, a missed merge) corrupts the whole structure.

This notebook implements the **hybrid pipeline** from `another_approach.md`. It splits the
job so no single stage has to do everything, and the final HTML is produced by a
**deterministic renderer** (never by the model), so the output is always valid HTML:

```
Image
  └─ Stage 1  OCR            → text + bounding boxes        (dedicated OCR)
  └─ Stage 2  Geometry       → row/col clusters, merge hints (pure numpy/opencv)
  └─ Stage 3  Structural VLM → cell-graph JSON               (BIG model, accuracy-critical)
  └─ Stage 4  Renderer       → <table> HTML                  (deterministic, in-code)
```

**Model policy (per your setup):**
- Everything accuracy-critical (OCR fallback + structural reasoning) uses the **big model**.
- The **small model** appears only as the fine-tune / few-shot *target* — it is **not** used to reconstruct here.
- Both are your own vLLM services, called through the OpenAI SDK at `api_url="http://localhost:3112"`.

Run top to bottom. Each stage is a plain function so you can inspect its output before the next one.

## 0. Install

Heavy deps are the OCR engine + OpenCV. Run this once. If you manage envs with `uv`,
prefer `uv pip install paddlepaddle paddleocr opencv-python-headless openai pillow numpy beautifulsoup4 lxml`
in a terminal instead of the cell below.

In [ ]:
# %pip install -q paddlepaddle paddleocr opencv-python-headless openai pillow numpy beautifulsoup4 lxml
# PaddleOCR pulls a lot in; if it is a problem you can skip it and use OCR_ENGINE="vlm" below.
print("install cell — uncomment to run")

## 1. Config — point these at YOUR served models

Only this cell is environment-specific. Set `API_URL` to your vLLM endpoint and the two
model names to whatever you serve.

In [ ]:
import os
from openai import OpenAI

# ── your vLLM OpenAI-compatible endpoint ─────────────────────────────
api_url   = os.environ.get("VLLM_API_URL", "http://localhost:3112")
api_key   = os.environ.get("VLLM_API_KEY", "EMPTY")   # vLLM ignores the value

# ── models ───────────────────────────────────────────────────────────
BIG_MODEL   = "Qwen3-VL-30B-A3B-FP8"        # accuracy-critical: OCR fallback + structural reasoning
SMALL_MODEL = "Qwen/Qwen3-VL-8B-Instruct"   # NOT used here; the future fine-tune / few-shot target

# ── OCR engine: "paddle" (dedicated, better bboxes) or "vlm" (big model, no paddle install) ──
OCR_ENGINE = "paddle"

client = OpenAI(base_url=f"{api_url}/v1", api_key=api_key)

# smoke test — uncomment when the endpoint is up
# print("served models:", [m.id for m in client.models.list().data])
print("config ready →", api_url, "| big:", BIG_MODEL)

## 2. Shared helpers (image encode, JSON extraction)

In [ ]:
import base64, json, re
from PIL import Image

def img_to_b64(path, max_side=1600):
    """Base64-encode, downscaling only if very large (protects the VLM ctx + your VRAM)."""
    im = Image.open(path).convert("RGB")
    if max(im.size) > max_side:
        s = max_side / max(im.size)
        im = im.resize((int(im.width * s), int(im.height * s)))
    from io import BytesIO
    buf = BytesIO(); im.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def extract_json(text):
    """Pull a JSON array/object out of a model reply, tolerating fences + stray prose."""
    t = text.strip()
    t = re.sub(r"^```(?:json)?", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    # prefer the outermost [...] (cell graph is an array)
    s, e = t.find("["), t.rfind("]")
    if s != -1 and e != -1 and e > s:
        t = t[s:e + 1]
    return json.loads(t)

## 3. Stage 1 — OCR (text + bounding boxes)

Dedicated OCR gives far more stable boxes than a VLM. PaddleOCR is the default; a big-model
fallback exists so you can run even without Paddle installed. Output is a list of
`{"text", "conf", "bbox":[x1,y1,x2,y2]}`.

In [ ]:
_paddle = None

def _run_ocr_paddle(image_path):
    global _paddle
    if _paddle is None:
        from paddleocr import PaddleOCR
        _paddle = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
    res = _paddle.ocr(image_path, cls=True)
    boxes = []
    lines = res[0] if res and res[0] else []
    for poly, (text, conf) in lines:
        xs = [p[0] for p in poly]; ys = [p[1] for p in poly]
        boxes.append({"text": text, "conf": float(conf),
                      "bbox": [min(xs), min(ys), max(xs), max(ys)]})
    return boxes

def _run_ocr_vlm(image_path):
    """Fallback: ask the big model for text+bbox. Bboxes are approximate — paddle is preferred."""
    b64 = img_to_b64(image_path)
    prompt = ("Extract every text token in this table with its pixel bounding box. "
              'Return JSON only: a list of {"text": str, "bbox": [x1,y1,x2,y2]}. '
              "Coordinates are pixels in the image. No prose, no HTML.")
    resp = client.chat.completions.create(
        model=BIG_MODEL, temperature=0.0, max_tokens=4000,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}])
    raw = extract_json(resp.choices[0].message.content)
    return [{"text": r["text"], "conf": 1.0, "bbox": r["bbox"]} for r in raw]

def run_ocr(image_path):
    return _run_ocr_paddle(image_path) if OCR_ENGINE == "paddle" else _run_ocr_vlm(image_path)

## 4. Stage 2 — Geometry (row/col clustering, merge hints)

Pure numpy. Cluster box centres into row bands and column bands (thresholds scale off the
median box size), then flag boxes that are much wider/taller than average as merge candidates.
These are only **hints** — the VLM in Stage 3 corrects them using the actual image.

In [ ]:
import numpy as np

def _cluster_1d(centers, threshold):
    """Group sorted 1-D centres: a gap larger than threshold starts a new cluster."""
    order = np.argsort(centers)
    labels = np.empty(len(centers), dtype=int)
    cur, prev = 0, None
    for i in order:
        if prev is not None and centers[i] - prev > threshold:
            cur += 1
        labels[i] = cur
        prev = centers[i]
    return labels

def build_geometry(boxes):
    if not boxes:
        return {"n_row_bands": 0, "n_col_bands": 0, "boxes": boxes}
    yc = np.array([(b["bbox"][1] + b["bbox"][3]) / 2 for b in boxes])
    xc = np.array([(b["bbox"][0] + b["bbox"][2]) / 2 for b in boxes])
    hs = np.array([b["bbox"][3] - b["bbox"][1] for b in boxes])
    ws = np.array([b["bbox"][2] - b["bbox"][0] for b in boxes])
    med_h, med_w = float(np.median(hs)), float(np.median(ws))

    row_lbl = _cluster_1d(yc, threshold=med_h * 0.6)
    col_lbl = _cluster_1d(xc, threshold=med_w * 0.5)

    # remap cluster ids to top→bottom / left→right order
    row_order = {c: i for i, c in enumerate(sorted(set(row_lbl), key=lambda c: yc[row_lbl == c].mean()))}
    col_order = {c: i for i, c in enumerate(sorted(set(col_lbl), key=lambda c: xc[col_lbl == c].mean()))}

    avg_w, avg_h = float(ws.mean()), float(hs.mean())
    for b, r, c in zip(boxes, row_lbl, col_lbl):
        b["row_candidate"] = row_order[r]
        b["col_candidate"] = col_order[c]
        bw, bh = b["bbox"][2] - b["bbox"][0], b["bbox"][3] - b["bbox"][1]
        b["candidate_colspan"] = bool(bw > 1.6 * avg_w)
        b["candidate_rowspan"] = bool(bh > 1.6 * avg_h)
    return {"n_row_bands": len(row_order), "n_col_bands": len(col_order), "boxes": boxes}

### 4b. Optional — ruling-line detection (OpenCV)

Many financial tables have explicit rules. Detecting them helps the VLM place borders/merges.
Optional; the pipeline runs without it.

In [ ]:
def detect_lines(image_path, min_len=40):
    import cv2
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return {"horizontal": 0, "vertical": 0}
    bw = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                               cv2.THRESH_BINARY_INV, 15, -2)
    h_k = cv2.getStructuringElement(cv2.MORPH_RECT, (min_len, 1))
    v_k = cv2.getStructuringElement(cv2.MORPH_RECT, (1, min_len))
    h = cv2.morphologyEx(bw, cv2.MORPH_OPEN, h_k)
    v = cv2.morphologyEx(bw, cv2.MORPH_OPEN, v_k)
    n_h = cv2.connectedComponents(h)[0] - 1
    n_v = cv2.connectedComponents(v)[0] - 1
    return {"horizontal": int(n_h), "vertical": int(n_v)}

## 5. Stage 3 — Structural reasoning (BIG model → cell-graph JSON)

The big model receives **image + OCR tokens + geometry hints** and returns a **cell graph**:
one object per logical cell with 0-based inclusive grid spans. It does **not** emit HTML — a
cell graph is far easier for a VLM to get right, and easy to validate/debug.

In [ ]:
STRUCT_SYS = "You are an expert at reconstructing the logical grid structure of tables."

def build_struct_prompt(boxes, geom, lines=None):
    tokens = [{"id": i, "text": b["text"],
               "row_candidate": b["row_candidate"], "col_candidate": b["col_candidate"],
               "candidate_colspan": b["candidate_colspan"],
               "candidate_rowspan": b["candidate_rowspan"]} for i, b in enumerate(boxes)]
    line_note = f"\nDetected ruling lines: {lines}." if lines else ""
    return f"""You are given a financial table:
1. The original image.
2. OCR tokens with text and APPROXIMATE candidate row/column indices (from clustering).
3. {geom['n_row_bands']} row bands and {geom['n_col_bands']} column bands were detected.{line_note}

OCR tokens (JSON):
{json.dumps(tokens, ensure_ascii=False)}

Task: infer the LOGICAL grid of the table. For EVERY cell in the grid output its text and
its 0-based INCLUSIVE span. Use the IMAGE to fix merged cells, multi-row headers, blank
cells, indentation grouping and alignment that the OCR candidates get wrong.

Return JSON ONLY — no prose, no HTML, no code fences. Schema: a JSON array of objects:
  {{"text": str, "row_start": int, "row_end": int, "col_start": int, "col_end": int, "is_header": bool}}

Rules:
- Single cell => row_start==row_end and col_start==col_end.
- A merged cell spanning 2 columns => col_end = col_start + 1, etc.
- Cover the FULL rectangular grid. Include blank cells as objects with "text": "".
- Spans must NOT overlap and must NOT leave gaps.
- Header rows (column titles / group headers) => "is_header": true."""

def infer_cell_graph(image_path, boxes, geom, lines=None, max_tokens=6000):
    b64 = img_to_b64(image_path)
    prompt = build_struct_prompt(boxes, geom, lines)
    resp = client.chat.completions.create(
        model=BIG_MODEL, temperature=0.0, max_tokens=max_tokens,
        messages=[{"role": "system", "content": STRUCT_SYS},
                  {"role": "user", "content": [
                      {"type": "text", "text": prompt},
                      {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}])
    return extract_json(resp.choices[0].message.content)

## 6. Stage 4 — Deterministic renderer + grid validation

The renderer turns the cell graph into HTML by replaying the grid: it walks every (row, col),
emits each cell once with the right `rowspan`/`colspan`, and fills any uncovered slot with an
empty `<td></td>`. `validate_grid` independently re-parses the HTML to prove it is rectangular
with no overlaps or gaps.

In [ ]:
import html as html_lib
from bs4 import BeautifulSoup

def cellgraph_to_html(cells):
    cells = [c for c in cells if c.get("col_end", -1) >= 0 and c.get("row_end", -1) >= 0]
    if not cells:
        return "<table></table>"
    n_rows = max(c["row_end"] for c in cells) + 1
    n_cols = max(c["col_end"] for c in cells) + 1
    occ = [[False] * n_cols for _ in range(n_rows)]
    starts = {(c["row_start"], c["col_start"]): c for c in cells}
    out = ["<table>"]
    for r in range(n_rows):
        out.append("<tr>")
        for cc in range(n_cols):
            if occ[r][cc]:
                continue
            c = starts.get((r, cc))
            if c is None:                      # uncovered slot => genuine blank cell
                out.append("<td></td>"); occ[r][cc] = True; continue
            rs = max(1, c["row_end"] - c["row_start"] + 1)
            cs = max(1, c["col_end"] - c["col_start"] + 1)
            for dr in range(rs):
                for dc in range(cs):
                    if r + dr < n_rows and cc + dc < n_cols:
                        occ[r + dr][cc + dc] = True
            tag = "th" if c.get("is_header") else "td"
            attrs = (f' colspan="{cs}"' if cs > 1 else "") + (f' rowspan="{rs}"' if rs > 1 else "")
            out.append(f"<{tag}{attrs}>{html_lib.escape((c.get('text') or '').strip())}</{tag}>")
        out.append("</tr>")
    out.append("</table>")
    return "\n".join(out)

def validate_grid(html):
    """Replay HTML layout; return (True,(rows,cols)) or (False, reason)."""
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table")
    if not table:
        return False, "no table"
    occ = {}; nrows = 0; ncols = 0
    for r, tr in enumerate(table.find_all("tr")):
        c = 0
        for cell in tr.find_all(["td", "th"]):
            while occ.get((r, c)):
                c += 1
            rs = int(cell.get("rowspan", 1)); cs = int(cell.get("colspan", 1))
            for dr in range(rs):
                for dc in range(cs):
                    if occ.get((r + dr, c + dc)):
                        return False, "overlap"
                    occ[(r + dr, c + dc)] = True
            c += cs; ncols = max(ncols, c)
        nrows = max(nrows, r + 1)
    for r in range(nrows):
        for cc in range(ncols):
            if not occ.get((r, cc)):
                return False, "gap"
    return True, (nrows, ncols)

## 7. End-to-end reconstruction

One call runs all four stages and returns every intermediate artifact so you can debug where
a table went wrong (bad OCR? bad clusters? bad cell graph?).

In [ ]:
def reconstruct(image_path, use_lines=False, verbose=True):
    boxes = run_ocr(image_path)
    geom  = build_geometry(boxes)
    lines = detect_lines(image_path) if use_lines else None
    cells = infer_cell_graph(image_path, boxes, geom, lines)
    html  = cellgraph_to_html(cells)
    ok, info = validate_grid(html)
    if verbose:
        tag = f"{info[0]}x{info[1]}" if ok else f"INVALID:{info}"
        print(f"{os.path.basename(image_path)}: {len(boxes)} tokens, "
              f"{geom['n_row_bands']}x{geom['n_col_bands']} bands, "
              f"{len(cells)} cells, grid {tag}")
    return {"image": image_path, "boxes": boxes, "geom": geom,
            "cells": cells, "html": html, "valid": ok, "grid": info}

## 8. Run it — side-by-side image vs reconstructed HTML

Uses the stand-in images in `val_mini/mini_val/`. Swap `SAMPLES` for your own paths.

In [ ]:
import glob
from IPython.display import HTML, display

SAMPLES = sorted(glob.glob("val_mini/mini_val/*.png"))[:3]

def show(result):
    b64 = img_to_b64(result["image"])
    badge = "✅ valid" if result["valid"] else f"⚠ {result['grid']}"
    display(HTML(f"""
    <div style="display:flex; gap:24px; align-items:flex-start; margin:12px 0;
                border-bottom:1px solid #ccc; padding-bottom:12px">
      <div><div style="font:12px monospace">{os.path.basename(result['image'])} &nbsp; {badge}</div>
           <img src="data:image/png;base64,{b64}" style="max-width:460px; border:1px solid #ddd"/></div>
      <div style="font:13px sans-serif">
           <div style="color:#666; font:12px monospace">reconstructed</div>
           <div style="border:1px solid #ddd; padding:6px">{result['html']}</div></div>
    </div>"""))

for path in SAMPLES:
    try:
        show(reconstruct(path))
    except Exception as e:
        print(f"{os.path.basename(path)}: FAILED — {type(e).__name__}: {e}")

## 9. (Optional) A/B — naive direct prompt vs this hybrid pipeline

Run the same image through the plain `image → HTML` prompt so you can *see* why the hybrid
pipeline wins on your hard tables. Both use the big model; only the method differs.

In [ ]:
def direct_image_to_html(image_path, max_tokens=6000):
    b64 = img_to_b64(image_path)
    prompt = ("Convert this table image to HTML. Output only a <table>...</table> using "
              "th/td with colspan/rowspan for merged cells. No prose, no code fences.")
    resp = client.chat.completions.create(
        model=BIG_MODEL, temperature=0.0, max_tokens=max_tokens,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}])
    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:html)?", "", raw).strip(); raw = re.sub(r"```$", "", raw).strip()
    return raw

def compare(image_path):
    hy = reconstruct(image_path, verbose=False)
    direct = direct_image_to_html(image_path)
    d_ok, d_info = validate_grid(direct)
    b64 = img_to_b64(image_path)
    display(HTML(f"""
    <div style="margin:12px 0; border-bottom:1px solid #ccc; padding-bottom:12px">
      <div style="font:12px monospace">{os.path.basename(image_path)}</div>
      <div style="display:flex; gap:20px; align-items:flex-start">
        <div><div style="font:11px monospace; color:#666">image</div>
             <img src="data:image/png;base64,{b64}" style="max-width:360px; border:1px solid #ddd"/></div>
        <div><div style="font:11px monospace; color:#666">naive direct &nbsp; {'✅' if d_ok else '⚠ '+str(d_info)}</div>
             <div style="border:1px solid #ddd; padding:6px; font:12px sans-serif">{direct}</div></div>
        <div><div style="font:11px monospace; color:#666">hybrid &nbsp; {'✅' if hy['valid'] else '⚠ '+str(hy['grid'])}</div>
             <div style="border:1px solid #ddd; padding:6px; font:12px sans-serif">{hy['html']}</div></div>
      </div></div>"""))

# for path in SAMPLES[:2]:
#     compare(path)

## 10. What this produces & where the small model comes in

This pipeline is your **reliable labeler**. For each image it yields validated HTML — perfect
gold labels for a training set, with every intermediate stage inspectable.

**Next (not in this notebook):**
- Batch-run `reconstruct` over your real finance images → save `{image, html}` JSONL as the eval/grounding set.
- Combine with the synthetic set from `synthetic-table-generator.ipynb`.
- QLoRA fine-tune **`SMALL_MODEL`** on that data (vision tower frozen, LoRA on LM projections).
- The small model then does the same task cheaply — and the hybrid pipeline stays available as a high-accuracy fallback.

The small model is **only** a fine-tune / few-shot target; all accuracy-critical work here stays on the big model.